# Classificador Focado / Distraído — Transfer Learning com MobileNetV2 (v2)

Notebook **educativo** para treinar um classificador binário de imagens
(**focused** vs **distracted**) com *transfer learning* usando **MobileNetV2**.

Esta é uma versão revisada do treino original. As mudanças principais em relação
à v1 (cada uma explicada na seção correspondente):

| # | Mudança | Por quê |
|---|---------|---------|
| 1 | **Split anti-leakage por grupo/pessoa** (opcional) | Evita que frames da mesma pessoa caiam em treino *e* teste, o que inflaria a acurácia artificialmente |
| 2 | **Callbacks** (`EarlyStopping`, `ReduceLROnPlateau`) | Para no ponto certo, restaura o melhor modelo e ajusta o LR sozinho |
| 3 | **Cabeça densa mais enxuta** | Menos parâmetros = menos overfit num dataset pequeno |
| 4 | **BatchNorm congelado no fine-tuning** | Mantém estável o ajuste fino da base |
| 5 | **`class_weight` + métricas além da acurácia** | Lida com classes desbalanceadas; precision/recall/AUC não enganam |
| 6 | **`.cache()` no pipeline de dados** | Treino bem mais rápido |
| 7 | **Análise de threshold** | O corte 0.5 nem sempre é o ideal |

> Execute as células **de cima para baixo, sem pular** — cada seção depende das variáveis da anterior.


## 0. Guia de instalação e execução (leia primeiro)

As imagens ficam na pasta **`data/`** do projeto, com as subpastas `focused/` e
`distracted/`. **Não usamos Google Drive.** Escolha **uma** das formas abaixo.

---
### Opção A — No seu computador (Jupyter local)
1. Instale Python + Jupyter.
2. No terminal, uma única vez:
   ```
   pip install tensorflow scikit-learn matplotlib numpy notebook
   ```
3. Abra o terminal **na raiz do projeto** (a pasta que contém `data/` e `notebooks/`) e rode `jupyter notebook`.
4. Abra `notebooks/train_v2.ipynb`. Sem GPU o treino roda na CPU e fica lento.

---
### Opção B — No Google Colab (GPU grátis)
1. Compacte a pasta `data` em **`data.zip`**.
2. <https://colab.research.google.com> → **Arquivo > Fazer upload de notebook** → este `.ipynb`.
3. **Ambiente de execução > Alterar tipo de ambiente > GPU > Salvar**.
4. Arraste `data.zip` para o painel de arquivos à esquerda.
5. Numa célula: `!unzip -q data.zip -d /content` (deve gerar `/content/data/focused` e `/content/data/distracted`).
6. Siga o notebook normalmente.

---
### Se aparecer `ModuleNotFoundError`
Confirme que rodou as células **na ordem**, rode o `pip install`, reinicie o kernel e rode tudo de novo.


## 1. Setup e imports

Importamos as bibliotecas, detectamos automaticamente se estamos no Colab ou local
e confirmamos a versão do TensorFlow e se há GPU. Cada import crítico imprime um `✓`.


In [ ]:
import os
import re
import shutil
import random
import glob

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

print(f"✓ TensorFlow {tf.__version__}")
print(f"✓ Keras {keras.__version__}")
print(f"✓ NumPy {np.__version__}")
print("✓ Matplotlib ok")

# ── Detectar ambiente (Colab x local). NÃO usamos Google Drive. ─────────
try:
    import google.colab  # só existe dentro do Colab
    EM_COLAB = True
    print("✓ Google Colab detectado")
except ImportError:
    EM_COLAB = False
    print("✓ Rodando LOCALMENTE — Google Drive não é necessário")

# ── GPU ─────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"✓ GPU disponível: {gpus}")
else:
    print("⚠ Sem GPU — treino funciona, mas mais lento (CPU).")

## 2. Configurações globais

Todas as constantes em um só lugar. Destaques desta versão:

- **`SPLIT_POR_GRUPO`** — se `True`, garante que todas as imagens de uma mesma
  *pessoa/sessão* fiquem no mesmo split (treino **ou** validação **ou** teste, nunca
  divididas). Isso evita o *data leakage* mais comum em projetos de webcam. Veja a
  Seção 3 para configurar o padrão do nome do arquivo.
- Imagens **RGB (3 canais)** — o MobileNetV2 foi pré-treinado em RGB. Não converta para cinza.
- `EARLYSTOP_PATIENCE` / `REDUCELR_PATIENCE` — controlam os callbacks da Seção 7.


In [ ]:
# ── Descobrir a pasta do projeto (a que contém 'data/focused') ─────────
def _achar_base():
    candidatos = [
        os.getcwd(),
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        "/content",
        "/content/focus_project",
    ]
    for c in candidatos:
        if os.path.isdir(os.path.join(c, "data", "focused")):
            return c
    return os.getcwd()

BASE_DIR      = _achar_base()
DATA_DIR      = os.path.join(BASE_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

IMG_SIZE   = (224, 224)               # entrada do MobileNetV2
BATCH_SIZE = 32
CLASSES    = ["focused", "distracted"]  # 0 = focused, 1 = distracted
SEED       = 42

# ── Anti data-leakage ───────────────────────────────────────────────────
# True  -> split por grupo (todas as imgs de uma pessoa/sessão no mesmo split)
# False -> split aleatório por arquivo (use só se cada imagem for de uma fonte distinta)
SPLIT_POR_GRUPO = True

# ── Controle dos callbacks (Seção 7) ────────────────────────────────────
EARLYSTOP_PATIENCE = 4   # épocas sem melhora em val_loss até parar
REDUCELR_PATIENCE  = 2   # épocas sem melhora até reduzir o learning rate

# Reprodutibilidade
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("Configurações:")
print(f"  BASE_DIR        = {BASE_DIR}")
print(f"  DATA_DIR        = {DATA_DIR}")
print(f"  MODELS_DIR      = {MODELS_DIR}")
print(f"  IMG_SIZE        = {IMG_SIZE} (RGB)")
print(f"  BATCH_SIZE      = {BATCH_SIZE}")
print(f"  CLASSES         = {CLASSES}")
print(f"  SPLIT_POR_GRUPO = {SPLIT_POR_GRUPO}")
print(f"  SEED            = {SEED}")

if not os.path.isdir(os.path.join(DATA_DIR, "focused")):
    print("\n⚠ Não encontrei", os.path.join(DATA_DIR, "focused"))
    print("  Ajuste BASE_DIR manualmente para a pasta que contém 'data/'.")
else:
    print("\n✓ Pasta de dados encontrada.")

## 3. Separação dos dados (train / val / test) — com proteção contra leakage

As fotos originais estão em duas pastas: `data/focused/` e `data/distracted/`.
O notebook faz **ele mesmo** a divisão 70/15/15.

### O problema do *data leakage* (por que isso importa)
Se as suas imagens forem **frames de webcam de poucas pessoas**, dividir
aleatoriamente *por arquivo* faz frames quase idênticos da **mesma pessoa** caírem
em treino **e** em teste. O modelo então "reconhece a pessoa" em vez de aprender o
que é estar focado — e a acurácia de teste fica **inflada e mentirosa**.

**Solução:** quando `SPLIT_POR_GRUPO = True`, agrupamos as imagens por *pessoa/sessão*
e mandamos cada grupo inteiro para um único split. A função `_extrair_grupo` decide
o grupo a partir do **nome do arquivo** via `GROUP_REGEX`.

> Ajuste o `GROUP_REGEX` ao seu padrão de nomes. Exemplos:
> - `aluno03_frame0125.jpg` → grupo `aluno03` (padrão atual: pega o trecho antes do último `_`/`-`/número).
> - Se cada imagem já for de uma pessoa diferente, ponha `SPLIT_POR_GRUPO = False` na Seção 2.

A célula é **idempotente**: se `data/processed/` já existir, ela avisa em vez de duplicar.


In [ ]:
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
SPLITS       = ["train", "val", "test"]
VALID_EXT    = (".jpg", ".jpeg", ".png", ".bmp")

# Regex que extrai o "grupo" (pessoa/sessão) do NOME do arquivo.
# Padrão: tudo até antes do último separador _ / - seguido de dígitos.
# Ex.: "aluno03_frame0125.jpg" -> "aluno03" ; "p12-007.png" -> "p12"
GROUP_REGEX = re.compile(r"^(.*?)[ _\-]*\d+\.[^.]+$")

def _extrair_grupo(caminho):
    nome = os.path.basename(caminho)
    m = GROUP_REGEX.match(nome)
    if m and m.group(1):
        return m.group(1)
    return os.path.splitext(nome)[0]  # fallback: o nome inteiro vira o grupo

def listar_imagens(pasta):
    arquivos = []
    for ext in VALID_EXT:
        arquivos += glob.glob(os.path.join(pasta, f"*{ext}"))
        arquivos += glob.glob(os.path.join(pasta, f"*{ext.upper()}"))
    return sorted(set(arquivos))

def dividir_por_arquivo(imagens):
    rng = random.Random(SEED); rng.shuffle(imagens)
    n = len(imagens)
    n_tr = int(n * SPLIT_RATIOS["train"]); n_va = int(n * SPLIT_RATIOS["val"])
    return {"train": imagens[:n_tr],
            "val":   imagens[n_tr:n_tr + n_va],
            "test":  imagens[n_tr + n_va:]}

def dividir_por_grupo(imagens):
    # agrupa por pessoa/sessão e distribui GRUPOS inteiros para cada split
    grupos = {}
    for img in imagens:
        grupos.setdefault(_extrair_grupo(img), []).append(img)
    chaves = sorted(grupos)
    rng = random.Random(SEED); rng.shuffle(chaves)
    g = len(chaves)
    g_tr = int(g * SPLIT_RATIOS["train"]); g_va = int(g * SPLIT_RATIOS["val"])
    sel = {"train": chaves[:g_tr],
           "val":   chaves[g_tr:g_tr + g_va],
           "test":  chaves[g_tr + g_va:]}
    return {s: [img for k in sel[s] for img in grupos[k]] for s in SPLITS}, len(chaves)

if os.path.exists(PROCESSED_DIR):
    print("⚠ 'data/processed/' já existe — pulando a divisão para não duplicar.")
    print("   Apague essa pasta manualmente se quiser refazer do zero.")
else:
    contagem = {s: {} for s in SPLITS}
    total_original = {}
    for classe in CLASSES:
        origem  = os.path.join(DATA_DIR, classe)
        imagens = listar_imagens(origem)
        total_original[classe] = len(imagens)
        if not imagens:
            raise FileNotFoundError(f"Nenhuma imagem em {origem}. Confira BASE_DIR.")

        if SPLIT_POR_GRUPO:
            divisao, n_grupos = dividir_por_grupo(imagens)
            print(f"[{classe}] {len(imagens)} imagens em {n_grupos} grupos (pessoas/sessões)")
        else:
            divisao = dividir_por_arquivo(imagens)

        for split in SPLITS:
            destino = os.path.join(PROCESSED_DIR, split, classe)
            os.makedirs(destino, exist_ok=True)
            for caminho in divisao[split]:
                shutil.copy2(caminho, destino)
            contagem[split][classe] = len(divisao[split])

    # ── VERIFICAÇÃO: tabela de contagem ────────────────────────────────
    print("\n=== VERIFICAÇÃO DA DIVISÃO ===")
    print(f"{'Classe':<12}{'train':>8}{'val':>8}{'test':>8}{'soma':>8}{'orig':>8}")
    print("-" * 52)
    for classe in CLASSES:
        soma = sum(contagem[s][classe] for s in SPLITS)
        print(f"{classe:<12}{contagem['train'][classe]:>8}{contagem['val'][classe]:>8}"
              f"{contagem['test'][classe]:>8}{soma:>8}{total_original[classe]:>8}")
        assert soma == total_original[classe], f"Soma não bate para {classe}!"
    print("-" * 52)
    print("✓ Somas batem. Divisão concluída.")
    if SPLIT_POR_GRUPO:
        print("✓ Split POR GRUPO: nenhuma pessoa/sessão aparece em mais de um split.")

## 4. Carregamento dos dados

Carregamos os três conjuntos com `image_dataset_from_directory` a partir de
`data/processed/`, forçando `class_names=CLASSES` para garantir **focused = 0** e
**distracted = 1** em todos os splits.

**Verificação:** grid 3×3 de imagens do treino com seus rótulos.


In [ ]:
train_dir = os.path.join(PROCESSED_DIR, "train")
val_dir   = os.path.join(PROCESSED_DIR, "val")
test_dir  = os.path.join(PROCESSED_DIR, "test")

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, labels="inferred", label_mode="binary", class_names=CLASSES,
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE, shuffle=True, seed=SEED)

val_ds = keras.utils.image_dataset_from_directory(
    val_dir, labels="inferred", label_mode="binary", class_names=CLASSES,
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE, shuffle=False)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="binary", class_names=CLASSES,
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE, shuffle=False)

class_names = train_ds.class_names  # guardar ANTES do prefetch
print("✓ Datasets carregados. Ordem das classes:", class_names)

In [ ]:
# ── VERIFICAÇÃO: grid 3×3 do treino ────────────────────────────────────
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i].numpy()[0])])
        plt.axis("off")
plt.suptitle("Exemplos do conjunto de TREINO", fontsize=14)
plt.tight_layout(); plt.show()

### Otimização do pipeline: `.cache()` + `.shuffle()` + `.prefetch()`

- **`.cache()`** mantém as imagens decodificadas na memória — a partir da 2ª época o
  treino fica bem mais rápido (o dataset de webcam costuma caber na RAM).
- **`.shuffle()`** (só no treino) reembaralha a cada época, melhorando a generalização.
- **`.prefetch()`** prepara o próximo lote enquanto a GPU processa o atual.

> O augmentation **não** entra aqui — ele vive dentro do modelo (Seção 6), então
> cachear o dataset cru é seguro e continua gerando variações novas a cada passo.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)
print("✓ cache + shuffle (treino) + prefetch aplicados.")

## 5. Data augmentation

Cria variações artificiais das imagens a cada época, ajudando a generalizar e reduzindo
overfitting. Como o dataset é pequeno, é uma das defesas mais importantes contra decorar
o treino — **mas o augmentation tem que respeitar o que define a classe.**

### Por que NÃO rotacionar (nem distorcer a pose)
Aqui a classe (focado vs distraído) depende muito da **pose da cabeça**: inclinação e
direção pra onde ela aponta. Um `RandomRotation` gira a imagem inteira no plano, então
transforma uma cabeça reta (foco) em cabeça torta — ensinando o modelo a ser *invariante*
à inclinação, que é exatamente o sinal que queremos que ele use. Isso **destrói sinal**
em vez de adicionar robustez. Por isso ele saiu daqui.

### O que mantivemos (varia sem mudar o rótulo)
- **`RandomFlip("horizontal")`** — foco/distração é simétrico: olhar pra esquerda ou pra
  direita é igualmente "distraído". O flip dobra a variedade de lados sem mentir sobre a classe.
- **`RandomTranslation(0.06)`** — a pessoa não fica perfeitamente centralizada na webcam.
  Translação pequena (não corta a cabeça) e preserva o ângulo.
- **`RandomZoom(0.10)`** — a distância à câmera varia; o ângulo da cabeça continua o mesmo.
- **`RandomBrightness` + `RandomContrast`** — webcam tem exposição automática e o ambiente
  varia muito de luz; é o tipo de ruído que o modelo precisa aprender a ignorar.

> Se notar que o flip atrapalha (ex.: viés sistemático, como o celular sempre na mesma mão),
> é só remover a primeira linha. Na dúvida, para foco/distração ele é seguro.

**Verificação:** a mesma imagem com 9 augmentations diferentes — confira que a cabeça
**nunca aparece girada**, só com luz/enquadramento diferentes.


In [ ]:
# A POSE da cabeça (inclinação + direção) é o principal sinal de foco vs distração,
# então NÃO usamos nada que altere o ângulo aparente da cabeça (ex.: RandomRotation
# giraria uma cabeça reta -> torta, ensinando o modelo a IGNORAR justo o que importa).
# Mantemos só variações que mudam SEM mudar o rótulo: iluminação e enquadramento.
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),                      # foco/distração é simétrico: virar p/ esq. ou dir. dá no mesmo
    keras.layers.RandomTranslation(0.06, 0.06, fill_mode="nearest"),  # pessoa descentralizada, sem cortar a cabeça
    keras.layers.RandomZoom(0.10, fill_mode="nearest"),         # distância à câmera varia; preserva o ângulo da cabeça
    keras.layers.RandomBrightness(0.20, value_range=(0, 255)),  # exposição automática da webcam varia
    keras.layers.RandomContrast(0.20),                          # contraste/iluminação de ambiente varia
], name="data_augmentation")

print("✓ Augmentation criado (preserva a pose da cabeça):")
for layer in data_augmentation.layers:
    print("   -", layer.__class__.__name__)

In [ ]:
# ── VERIFICAÇÃO: mesma imagem, 9 augmentations ─────────────────────────
for images, _ in train_ds.take(1):
    imagem = images[0]
    break

plt.figure(figsize=(9, 9))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    aug = data_augmentation(tf.expand_dims(imagem, 0), training=True)
    plt.imshow(tf.clip_by_value(aug[0], 0, 255).numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Mesma imagem com diferentes augmentations", fontsize=14)
plt.tight_layout(); plt.show()

## 6. Construção do modelo

*Transfer learning* com **MobileNetV2** congelado. A grande mudança aqui é a
**cabeça (classificador) mais enxuta**.

```
Input (224, 224, 3)
  → data_augmentation
  → mobilenet_v2.preprocess_input        # escala para [-1, 1], exigido pela base
  → MobileNetV2(include_top=False, weights="imagenet", trainable=False)
  → GlobalAveragePooling2D               # 1280 features
  → Dropout(0.3)
  → Dense(128) + ReLU
  → Dropout(0.3)
  → Dense(1)  + Sigmoid
```

### Por que enxugar a cabeça?
A v1 usava `Dense(256) → Dense(64)`, ~340 mil parâmetros treináveis em cima das 1280
features do MobileNet. Num dataset pequeno isso é capacidade demais e **convida o
overfit**: o modelo decora o treino. Uma única camada `Dense(128)` com dropout
costuma generalizar melhor e treina mais rápido. Se ainda assim houver overfit forte,
dá para reduzir para `Dense(64)` ou ir direto de `GlobalAveragePooling → Dense(1)`.

**Verificação:** `model.summary()` e a razão treináveis/congelados.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import mobilenet_v2

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,),
                         include_top=False, weights="imagenet")
base_model.trainable = False  # Estágio 1: base congelada

inputs  = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="focus_classifier")
model.summary()

In [ ]:
# ── VERIFICAÇÃO: treináveis vs congelados ──────────────────────────────
treinaveis = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
congelados = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
total = treinaveis + congelados
print(f"Treináveis: {treinaveis:,}")
print(f"Congelados: {congelados:,}")
print(f"Total:      {total:,}")
print(f"\n→ Só {100*treinaveis/total:.1f}% dos parâmetros treinam no Estágio 1.")

## 7. Callbacks, métricas e `class_weight`

Três ingredientes que faltavam na v1 e que mais melhoram o resultado:

**Callbacks** (criados frescos em cada estágio):
- `EarlyStopping(monitor="val_loss", restore_best_weights=True)` — para quando a
  validação para de melhorar e **restaura os pesos da melhor época** (você não fica
  com o modelo do último passo, possivelmente já em overfit).
- `ReduceLROnPlateau` — reduz o learning rate quando a validação estaciona, ajudando
  a "assentar" o treino sem você ter que adivinhar o LR.

**Métricas além de accuracy:** em problema binário (e ainda mais se desbalanceado),
accuracy engana. Acompanhamos também **precision**, **recall** e **AUC**. Damos nomes
fixos às métricas para os históricos dos dois estágios concatenarem direitinho.

**`class_weight`:** se houver muito mais imagens de uma classe que da outra, o modelo
tende a "chutar" a classe maior. O `class_weight` dá mais peso à classe minoritária no
cálculo da loss, compensando o desbalanceamento.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# ── Métricas nomeadas (nomes estáveis entre os dois estágios) ──────────
METRICS = [
    keras.metrics.BinaryAccuracy(name="accuracy"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
    keras.metrics.AUC(name="auc"),
]

# ── class_weight a partir da contagem de arquivos no treino ────────────
contagem_treino = {
    i: len(listar_imagens(os.path.join(train_dir, c)))
    for i, c in enumerate(CLASSES)
}
y_treino = np.concatenate([[i] * n for i, n in contagem_treino.items()])
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_treino)
class_weight = {0: float(pesos[0]), 1: float(pesos[1])}

print("Imagens de treino por classe:")
for i, c in enumerate(CLASSES):
    print(f"  {c:<11} (classe {i}): {contagem_treino[i]}")
print(f"\nclass_weight = {class_weight}")
if abs(class_weight[0] - class_weight[1]) < 0.1:
    print("→ Classes bem balanceadas; o class_weight terá pouco efeito (tudo bem).")
else:
    print("→ Classes desbalanceadas; o class_weight vai compensar a classe minoritária.")

def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLYSTOP_PATIENCE,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=REDUCELR_PATIENCE,
            min_lr=1e-7, verbose=1),
    ]

print("\n✓ Métricas, class_weight e fábrica de callbacks prontos.")

## 8. Treino — Estágio 1 (feature extraction)

Com a base **congelada**, treinamos só a cabeça que adicionamos. Usamos **Adam
(lr=1e-3)**, loss `binary_crossentropy` e as métricas da Seção 7.

Definimos um teto de épocas alto (`EPOCHS_STAGE1 = 15`), mas o **`EarlyStopping`
provavelmente para antes** — é ele quem decide o ponto certo, não um número fixo.


In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="binary_crossentropy",
              metrics=METRICS)

EPOCHS_STAGE1 = 15
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weight,
    callbacks=make_callbacks(),
)
epocas_estagio1 = len(history1.history["loss"])
print(f"✓ Estágio 1 concluído em {epocas_estagio1} épocas (early stopping pode ter cortado).")

## 9. Treino — Estágio 2 (fine-tuning)

Agora **descongelamos as últimas ~30 camadas** do MobileNetV2 e treinamos com um
learning rate baixo para ajustar a base sem destruir o que ela já sabe.

### Detalhe que a v1 não tinha: manter BatchNorm congelado
As camadas `BatchNormalization` guardam estatísticas (média/variância) aprendidas na
ImageNet. Se você as descongela, elas voltam a atualizar essas estatísticas com os seus
poucos lotes — e isso costuma **desestabilizar** o fine-tuning. A prática recomendada é
descongelar as camadas convolucionais das últimas posições **mas manter as BatchNorm em
modo inferência**. É o que o laço abaixo faz.

LR do fine-tuning: `2e-5` (entre o 1e-5 conservador e um 5e-5 mais agressivo). Com o
`ReduceLROnPlateau` ativo, não precisa ser exato.


In [ ]:
base_model.trainable = True

N_DESCONGELAR = 30
# Descongela as últimas N camadas, EXCETO as BatchNormalization
for layer in base_model.layers[:-N_DESCONGELAR]:
    layer.trainable = False
for layer in base_model.layers[-N_DESCONGELAR:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False   # BN fica em modo inferência
    else:
        layer.trainable = True

n_bn_congeladas = sum(
    1 for l in base_model.layers[-N_DESCONGELAR:]
    if isinstance(l, keras.layers.BatchNormalization))
print(f"Camadas no MobileNetV2: {len(base_model.layers)}")
print(f"Descongeladas: últimas {N_DESCONGELAR} (menos {n_bn_congeladas} BatchNorm que ficam congeladas)")

# Recompilar é OBRIGATÓRIO após mudar trainable
model.compile(optimizer=keras.optimizers.Adam(learning_rate=2e-5),
              loss="binary_crossentropy",
              metrics=METRICS)

treinaveis = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
print(f"Parâmetros treináveis agora: {treinaveis:,}")

In [ ]:
EPOCHS_STAGE2 = 15
total_epochs  = epocas_estagio1 + EPOCHS_STAGE2

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=epocas_estagio1,
    class_weight=class_weight,
    callbacks=make_callbacks(),
)

# ── Concatenar os históricos dos dois estágios ─────────────────────────
history = {}
for chave in history1.history:
    if chave in history2.history:
        history[chave] = history1.history[chave] + history2.history[chave]

print("✓ Estágio 2 (fine-tuning) concluído.")
print(f"✓ Histórico combinado: {len(history['loss'])} épocas no total.")

## 10. Visualização do treino

Quatro painéis: **loss**, **accuracy**, **precision/recall** e **AUC** (treino vs
validação). A linha tracejada marca o início do fine-tuning.

**Como ler:** se a curva de treino sobe e a de validação desce/estaciona, é overfit —
considere mais dropout, augmentation mais forte ou cabeça menor. Se as duas ainda sobem
no fim, dá para treinar mais épocas.


In [ ]:
epocas_estagio1 = len(history1.history["loss"])
x = range(1, len(history["loss"]) + 1)

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

ax[0,0].plot(x, history["loss"], marker="o", label="Treino")
ax[0,0].plot(x, history["val_loss"], marker="o", label="Validação")
ax[0,0].set_title("Loss"); ax[0,0].set_xlabel("Época"); ax[0,0].legend(); ax[0,0].grid(alpha=0.3)

ax[0,1].plot(x, history["accuracy"], marker="o", label="Treino")
ax[0,1].plot(x, history["val_accuracy"], marker="o", label="Validação")
ax[0,1].set_title("Accuracy"); ax[0,1].set_xlabel("Época"); ax[0,1].legend(); ax[0,1].grid(alpha=0.3)

ax[1,0].plot(x, history["val_precision"], marker="o", label="Precision (val)")
ax[1,0].plot(x, history["val_recall"], marker="o", label="Recall (val)")
ax[1,0].set_title("Precision / Recall (validação)"); ax[1,0].set_xlabel("Época"); ax[1,0].legend(); ax[1,0].grid(alpha=0.3)

ax[1,1].plot(x, history["auc"], marker="o", label="Treino")
ax[1,1].plot(x, history["val_auc"], marker="o", label="Validação")
ax[1,1].set_title("AUC"); ax[1,1].set_xlabel("Época"); ax[1,1].legend(); ax[1,1].grid(alpha=0.3)

for a in ax.flat:
    a.axvline(epocas_estagio1 + 0.5, color="gray", linestyle="--")
plt.tight_layout(); plt.show()

## 11. Avaliação no conjunto de teste

Desempenho final no **teste** (imagens nunca vistas). Coletamos as probabilidades uma
única vez e reaproveitamos para tudo.

Verificações:
- Acurácia/AUC finais (`model.evaluate`)
- **Matriz de confusão**
- **Classification report** (precision, recall, f1)
- Grid 3×3 com predição vs real (erros em vermelho)


In [ ]:
resultados = model.evaluate(test_ds, verbose=0, return_dict=True)
print("✓ Métricas finais no TESTE:")
for k, v in resultados.items():
    print(f"   {k:<10}: {v:.4f}")

In [ ]:
# ── Coletar probabilidades e rótulos do teste (uma vez só) ─────────────
y_true, y_prob = [], []
for images, labels in test_ds:
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())
    y_true.extend(labels.numpy().flatten().astype(int).tolist())
y_true = np.array(y_true); y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)
print(f"✓ {len(y_true)} imagens de teste avaliadas.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predito"); ax.set_ylabel("Real"); ax.set_title("Matriz de Confusão (limiar 0.5)")
limiar = cm.max() / 2
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > limiar else "black", fontsize=14)
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

In [ ]:
# ── Grid 3×3: predição vs real (erros em vermelho) ─────────────────────
for images, labels in test_ds.take(1):
    probs = model.predict(images, verbose=0)
    plt.figure(figsize=(9, 9))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        real = int(labels[i].numpy()[0]); pred = int(probs[i][0] >= 0.5)
        cor = "green" if real == pred else "red"
        plt.title(f"real: {CLASSES[real]}\npred: {CLASSES[pred]}", color=cor, fontsize=10)
        plt.axis("off")
    plt.suptitle("Predição vs Real (erros em vermelho)", fontsize=14)
    plt.tight_layout(); plt.show()
    break

### 11.1 (Opcional) Ajuste de threshold

O corte padrão de **0.5** assume que errar para um lado é tão ruim quanto para o outro
e que as classes são balanceadas. Nem sempre é o caso. Abaixo varremos vários cortes e
mostramos qual **maximiza o F1** na validação — você pode adotá-lo no teste se fizer
sentido para o seu objetivo (ex.: priorizar pegar todas as distrações = mais recall).

> Boas práticas: escolha o threshold olhando a **validação** e só então reporte no
> **teste**, para não "espiar" o teste ao calibrar.


In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

# probabilidades na VALIDAÇÃO
yv_true, yv_prob = [], []
for images, labels in val_ds:
    yv_prob.extend(model.predict(images, verbose=0).flatten().tolist())
    yv_true.extend(labels.numpy().flatten().astype(int).tolist())
yv_true = np.array(yv_true); yv_prob = np.array(yv_prob)

thresholds = np.linspace(0.05, 0.95, 19)
f1s = [f1_score(yv_true, (yv_prob >= t).astype(int), zero_division=0) for t in thresholds]
melhor_t = float(thresholds[int(np.argmax(f1s))])

plt.figure(figsize=(7, 4))
plt.plot(thresholds, f1s, marker="o")
plt.axvline(melhor_t, color="red", linestyle="--", label=f"melhor F1 @ {melhor_t:.2f}")
plt.axvline(0.5, color="gray", linestyle=":", label="0.50 (padrão)")
plt.xlabel("Threshold"); plt.ylabel("F1 (validação)"); plt.title("F1 vs Threshold")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Melhor threshold por F1 na validação: {melhor_t:.2f}")
print("\nNo TESTE com esse threshold:")
print(classification_report(y_true, (y_prob >= melhor_t).astype(int),
                            target_names=CLASSES, digits=4))

## 12. Salvar o modelo

Salvamos em `models/focus_model.keras` (formato moderno recomendado pelo Keras 3).
Como o `EarlyStopping` usou `restore_best_weights=True`, **o modelo salvo é o da melhor
época de validação** — não o do último passo.


In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
caminho_modelo = os.path.join(MODELS_DIR, "focus_model.keras")
model.save(caminho_modelo)
print("✓ Modelo salvo!")
print(f"  Caminho: {caminho_modelo}")